# Feature 1: Sentence Structure
Token Counts
Part of Speech (POS) prevalence

In [ ]:
# 1. Install Dependencies
%pip install sudachipy sudachidict_core pandas

In [6]:
# 2. Imports
import os
from pathlib import Path
import pandas as pd

In [7]:
# 3. Load Dataset
# Works from any notebook in the /notebooks folder
PROJECT_ROOT = Path().resolve().parent

file_path = PROJECT_ROOT / "data" / "raw" / "yahoo_questions.csv"

assert file_path.exists(), (
    f"File not found: {file_path}\n"
    "Make sure you're running this notebook from the /notebooks directory "
    "and the data file exists."
)

df = pd.read_csv(file_path)

df.head()

,id,question,human_answer,ai_answer
0,1,ライブにヘッドドレスをつけていくのは非常識でしょうか？,アイドルのライブでしたらそういう人もいるかもしれませんが、バンド系のライブでしょうか？\r\...,必ずしも非常識とは言えませんが、ヘッドドレスの大きさやデザイン、会場の環境によっては周囲への...
1,2,塗料にはなんでシンナーが必要なんでしょうか？,基本的な塗料というのは原液の様に濃く作られているからです。\r\n\r\n子供の頃に学校で使...,塗料にシンナーが必要なのは、簡単に言うと**「塗りやすくするため」と「塗膜をきれいに作るため...
2,3,公園でたまに遭遇する放置子、みなさまはどう対応してますか？\r,＞みなさんは放置子に付き纏われたりした際、付き合ってあげますか？\r\n\r\n以前の私なら...,公園で見かける「放置子」への対応は、人によってかなり分かれますが、共通しているのは**「親代...
3,4,現実にはありえないと思うんですけど、「もし」イギリスがユーロ加盟したら、コインの裏のデザイン...,EUに加盟することとEURを使うことは同じではありませんが\r\nEU再加盟の機運は高くなっ...,面白い仮定ですね。現実にはかなり可能性が低いですが、「もしイギリスがユーロに加盟したら」とい...
4,5,あなたの一番行きたい国はどこですか？なければ日本の場所や、外国の場所でもいいですよ。また、行...,南アフリカ共和国およびその周辺の国々。\r\n\r\nアフリカの大自然の中で人間に保護されな...,私は実際に「行きたい」という欲求は持たないのですが、もし好奇心だけで場所を選べるなら、かなり...


In [11]:
# 4. Initialize SudachiPy
from sudachipy import Dictionary

tokenizer = Dictionary().create()

In [12]:
# 5. Sentence Splitter
import re

def split_sentences(text):
    if pd.isna(text):
        return []
    
    sentences = re.split(r'[。！？]+', text)
    return [s.strip() for s in sentences if s.strip()]

In [13]:
# 5.1 Test Sentence Splitter
sample = long_df["text"].iloc[0]

print("RAW:")
print(sample)

print("\nSENTENCES:")
print(split_sentences(sample))

RAW:
アイドルのライブでしたらそういう人もいるかもしれませんが、バンド系のライブでしょうか？
めちゃくちゃ悪目立ちすると思います。

SENTENCES:
['アイドルのライブでしたらそういう人もいるかもしれませんが、バンド系のライブでしょうか', 'めちゃくちゃ悪目立ちすると思います']


In [14]:
# 5.2 Apply Sentence Splitting
long_df["sentences"] = long_df["text"].apply(split_sentences)

In [15]:
# 6. Sentence Token Length Function
def sentence_token_lengths(text):
    sentences = split_sentences(text)
    
    lengths = []
    for sent in sentences:
        tokens = [m.surface() for m in tokenizer.tokenize(sent)]
        lengths.append(len(tokens))
    
    return lengths

In [16]:
# 7. Apply token lengths
long_df["sentence_token_lengths"] = long_df["text"].apply(sentence_token_lengths)

In [17]:
# 8. Debug Inspection
for i in range(3):
    print("TEXT:", long_df["text"].iloc[i])
    print("SENTENCES:", long_df["sentences"].iloc[i])
    print("TOKEN LENGTHS:", long_df["sentence_token_lengths"].iloc[i])
    print("-" * 50)

TEXT: アイドルのライブでしたらそういう人もいるかもしれませんが、バンド系のライブでしょうか？
めちゃくちゃ悪目立ちすると思います。
SENTENCES: ['アイドルのライブでしたらそういう人もいるかもしれませんが、バンド系のライブでしょうか', 'めちゃくちゃ悪目立ちすると思います']
TOKEN LENGTHS: [23, 7]
--------------------------------------------------
TEXT: 基本的な塗料というのは原液の様に濃く作られているからです。

子供の頃に学校で使った絵の具や、馴染みのあるカルピスをそのまま使ったり、飲む方はいませんね、
必ず水で溶いたり、希釈をして使ったり飲んだりしているわけです。
SENTENCES: ['基本的な塗料というのは原液の様に濃く作られているからです', '子供の頃に学校で使った絵の具や、馴染みのあるカルピスをそのまま使ったり、飲む方はいませんね、\r\n必ず水で溶いたり、希釈をして使ったり飲んだりしているわけです']
TOKEN LENGTHS: [18, 48]
--------------------------------------------------
TEXT: ＞みなさんは放置子に付き纏われたりした際、付き合ってあげますか？

以前の私なら付き合ってあげたと思います。
でもいろいろあって今ではすっかり放置子とその親が苦手になったので、今の私なら1秒も付き合ってあげません。
すぐに放置子から離れます。
SENTENCES: ['＞みなさんは放置子に付き纏われたりした際、付き合ってあげますか', '以前の私なら付き合ってあげたと思います', 'でもいろいろあって今ではすっかり放置子とその親が苦手になったので、今の私なら1秒も付き合ってあげません', 'すぐに放置子から離れます']
TOKEN LENGTHS: [19, 11, 34, 7]
--------------------------------------------------


In [ ]:
# 9. Tokenization Function
# Surface tokens only.
def tokenize(text):
    if pd.isna(text):
        return []
    return [m.surface() for m in tokenizer.tokenize(text)]

def inspect_tokens(text):
    tokens = tokenizer.tokenize(text)
    
    for m in tokens:
        print(
            "SURFACE:", m.surface(),
            "| BASE:", m.dictionary_form(),
            "| POS:", m.part_of_speech()
        )


In [ ]:
# 10. Apply tokenization to both columns
df["human_tokens"] = df["human_answer"].apply(tokenize)
df["ai_tokens"] = df["ai_answer"].apply(tokenize)

In [ ]:
# 11. Token counts
df["human_token_count"] = df["human_tokens"].apply(len)
df["ai_token_count"] = df["ai_tokens"].apply(len)

# Quick sanity check:
df[["id", "human_token_count", "ai_token_count"]].head()

,id,human_token_count,ai_token_count
0,1,33,31
1,2,69,75
2,3,79,51
3,4,121,92
4,5,51,109


In [ ]:
# 12. Convert to long format 
human_df = df[["id", "human_answer", "human_tokens", "human_token_count"]].copy()
human_df["label"] = "human"
human_df = human_df.rename(columns={
    "human_answer": "text",
    "human_tokens": "tokens",
    "human_token_count": "token_count"
})

ai_df = df[["id", "ai_answer", "ai_tokens", "ai_token_count"]].copy()
ai_df["label"] = "ai"
ai_df = ai_df.rename(columns={
    "ai_answer": "text",
    "ai_tokens": "tokens",
    "ai_token_count": "token_count"
})

long_df = pd.concat([human_df, ai_df], ignore_index=True)

long_df.head()

# | id | text | tokens | token_count | label |

,id,text,tokens,token_count,label
0,1,アイドルのライブでしたらそういう人もいるかもしれませんが、バンド系のライブでしょうか？\r\...,"[アイドル, の, ライブ, でし, たら, そう, いう, 人, も, いる, か, も,...",33,human
1,2,基本的な塗料というのは原液の様に濃く作られているからです。\r\n\r\n子供の頃に学校で使...,"[基本的, な, 塗料, と, いう, の, は, 原液, の, 様, に, 濃く, 作ら,...",69,human
2,3,＞みなさんは放置子に付き纏われたりした際、付き合ってあげますか？\r\n\r\n以前の私なら...,"[＞, みな, さん, は, 放置, 子, に, 付き纏わ, れ, たり, し, た, 際,...",79,human
3,4,EUに加盟することとEURを使うことは同じではありませんが\r\nEU再加盟の機運は高くなっ...,"[EU, に, 加盟, する, こと, と, EUR, を, 使う, こと, は, 同じ, ...",121,human
4,5,南アフリカ共和国およびその周辺の国々。\r\n\r\nアフリカの大自然の中で人間に保護されな...,"[南アフリカ共和国, および, その, 周辺, の, 国々, 。, \r\n\r\n, アフ...",51,human


In [ ]:
# 13. Basic sanity stats
long_df.groupby("label")["token_count"].describe()

,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
ai,50.0,30.2,78.701710,0.0,0.0,0.0,0.0,420.0
human,50.0,18.2,44.463239,0.0,0.0,0.0,0.0,240.0


In [ ]:
# 14. Optional: inspect tokenization quality
for i in range(3):
    print("TEXT:", long_df["text"].iloc[i])
    print("-" * 50)
    inspect_tokens(long_df["text"].iloc[i])
    print("=" * 50)

TEXT: アイドルのライブでしたらそういう人もいるかもしれませんが、バンド系のライブでしょうか？
めちゃくちゃ悪目立ちすると思います。
--------------------------------------------------
SURFACE: アイドル | BASE: アイドル | POS: ('名詞', '普通名詞', '一般', '*', '*', '*')
SURFACE: の | BASE: の | POS: ('助詞', '格助詞', '*', '*', '*', '*')
SURFACE: ライブ | BASE: ライブ | POS: ('名詞', '普通名詞', '一般', '*', '*', '*')
SURFACE: でし | BASE: です | POS: ('助動詞', '*', '*', '*', '助動詞-デス', '連用形-一般')
SURFACE: たら | BASE: た | POS: ('助動詞', '*', '*', '*', '助動詞-タ', '仮定形-一般')
SURFACE: そう | BASE: そう | POS: ('副詞', '*', '*', '*', '*', '*')
SURFACE: いう | BASE: いう | POS: ('動詞', '一般', '*', '*', '五段-ワア行', '連体形-一般')
SURFACE: 人 | BASE: 人 | POS: ('名詞', '普通名詞', '一般', '*', '*', '*')
SURFACE: も | BASE: も | POS: ('助詞', '係助詞', '*', '*', '*', '*')
SURFACE: いる | BASE: いる | POS: ('動詞', '非自立可能', '*', '*', '上一段-ア行', '終止形-一般')
SURFACE: か | BASE: か | POS: ('助詞', '副助詞', '*', '*', '*', '*')
SURFACE: も | BASE: も | POS: ('助詞', '係助詞', '*', '*', '*', '*')
SURFACE: しれ | BASE: しれる | POS: ('動詞', '一般', '*', '*', '下一段-ラ行', '連用形-一般')
S

In [ ]:
# 15. (Optional but recommended) Flatten tokens for future linguistic features
# This makes later feature engineering easier.
token_rows = []

for _, row in long_df.iterrows():
    for tok in row["tokens"]:
        token_rows.append({
            "id": row["id"],
            "label": row["label"],
            "token": tok
        })

token_df = pd.DataFrame(token_rows)

token_df.head()

,id,label,token
0,1,human,アイドル
1,1,human,の
2,1,human,ライブ
3,1,human,でし
4,1,human,たら


# Feature 2: Connective Density (logical scaffolding)
This measures how often a text uses logical connectors like:

しかし (however)
そのため (therefore)
つまり (in other words)
一方で (on the other hand)

In Japanese, this is often a very strong signal between:
- human implicit reasoning
- AI explicit “essay-style” reasoning


Human hypothesis: 
- implicit reasoning
- fewer explicit logical markers
- more “jumping between ideas”

AI hypothesis:
- explicit discourse structure
- more connective-heavy explanations
- more “essay logic scaffolding”

In [24]:
# 2.1 Define Connective Lexicon
connectives = {
    "しかし", "ただし", "そのため", "したがって", "つまり",
    "一方で", "また", "さらに", "なお", "ゆえに", "なので"
}

In [25]:
# 2.2 Token-based Connective Extractor
def extract_connectives(text):
    if pd.isna(text):
        return []
    
    tokens = tokenizer.tokenize(text)
    
    found = []
    for m in tokens:
        surface = m.surface()
        base = m.dictionary_form()
        
        if surface in connectives or base in connectives:
            found.append(surface)
    
    return found

In [26]:
# 2.3 Apply to Dataset
long_df["connectives"] = long_df["text"].apply(extract_connectives)

In [27]:
# 2.4 Compute Connective Counts
long_df["connective_count"] = long_df["connectives"].apply(len)

In [28]:
# 2.5 Normalize
# Raw counts are misleading because longer text = more connectives automatically.
# This gives connectives per token.
long_df["connective_density"] = long_df["connective_count"] / long_df["token_count"].replace(0, 1)

In [29]:
# 2.6 Compare Human vs. AI
long_df.groupby("label")[[
    "connective_count",
    "connective_density"
]].describe()

connective_count                                           \
                 count  mean       std  min  25%  50%  75%  max   
label                                                             
ai                50.0  0.10  0.416497  0.0  0.0  0.0  0.0  2.0   
human             50.0  0.04  0.197949  0.0  0.0  0.0  0.0  1.0   

      connective_density                                                    
                   count      mean       std  min  25%  50%  75%       max  
label                                                                       
ai                  50.0  0.000432  0.001781  0.0  0.0  0.0  0.0  0.008621  
human               50.0  0.000588  0.003075  0.0  0.0  0.0  0.0  0.019608

In [30]:
# 2.7 Quick Inspection
for i in range(5):
    print("TEXT:", long_df["text"].iloc[i])
    print("CONNECTIVES:", long_df["connectives"].iloc[i])
    print("COUNT:", long_df["connective_count"].iloc[i])
    print("-" * 50)

TEXT: アイドルのライブでしたらそういう人もいるかもしれませんが、バンド系のライブでしょうか？
めちゃくちゃ悪目立ちすると思います。
CONNECTIVES: []
COUNT: 0
--------------------------------------------------
TEXT: 基本的な塗料というのは原液の様に濃く作られているからです。

子供の頃に学校で使った絵の具や、馴染みのあるカルピスをそのまま使ったり、飲む方はいませんね、
必ず水で溶いたり、希釈をして使ったり飲んだりしているわけです。
CONNECTIVES: []
COUNT: 0
--------------------------------------------------
TEXT: ＞みなさんは放置子に付き纏われたりした際、付き合ってあげますか？

以前の私なら付き合ってあげたと思います。
でもいろいろあって今ではすっかり放置子とその親が苦手になったので、今の私なら1秒も付き合ってあげません。
すぐに放置子から離れます。
CONNECTIVES: []
COUNT: 0
--------------------------------------------------
TEXT: EUに加盟することとEURを使うことは同じではありませんが
EU再加盟の機運は高くなっています
ただすぐにどうかは疑問ですが
すくなくても失敗したかも
ヨーロッパもかつてのイギリスの移民問題を理解してくれていそう
というのがあり、若い世代は８割再加盟に賛成のようです

かりに再加盟しEURを使うときのコインはおそらく
表面はEU共通デザイン
裏面は国別なので
国王か王室紋章などでしょうね
CONNECTIVES: []
COUNT: 0
--------------------------------------------------
TEXT: 南アフリカ共和国およびその周辺の国々。

アフリカの大自然の中で人間に保護されながら生息しているビッグ5(ヒョウ・ライオン・サイ・アフリカゾウ・カバ)を観たい。またビクトリア大瀑布を見学したいから。
CONNECTIVES: ['また']
COUNT: 1
----------------------------------------------

# Feature 3: Sentence Ending Diversity (Sylistic variation - very strong in Japanese)

We will measure:

です / ます patterns
と思います usage
rhetorical endings (でしょう / かもしれない)

This one tends to separate:

AI “polite uniformity”
vs
human stylistic variability


In [31]:
# 3.1 Define sentence ending extractor
def extract_sentence_endings(text):
    sentences = split_sentences(text)
    
    endings = []
    for s in sentences:
        s = s.strip()
        if not s:
            continue
        
        # take last 5–10 characters as a rough ending window
        endings.append(s[-10:])
    
    return endings

In [32]:
# 3.2 Apply to dataset
long_df["sentence_endings"] = long_df["text"].apply(extract_sentence_endings)

In [33]:
# 3.3 Normalize endings
def normalize_ending(e):
    return re.sub(r'[。！？\s]+', '', e)

long_df["normalized_endings"] = long_df["sentence_endings"].apply(
    lambda lst: [normalize_ending(e) for e in lst]
)

In [34]:
# 3.4 Compute diversity
# How many unique endings appear per text
def ending_diversity(endings):
    if len(endings) == 0:
        return 0
    return len(set(endings)) / len(endings)

# Apply
long_df["ending_diversity"] = long_df["normalized_endings"].apply(ending_diversity)

In [37]:
# 3.5 Entropy-style diversity compute
# This measures how distributed the sentence endings are. Instead of counting variety, it measures unpredictability.
from collections import Counter
import numpy as np

def ending_entropy(endings):
    if len(endings) == 0:
        return 0
    
    counts = Counter(endings)
    total = sum(counts.values())
    
    probs = [c / total for c in counts.values()]
    return -sum(p * np.log2(p) for p in probs)

# Apply
long_df["ending_entropy"] = long_df["normalized_endings"].apply(ending_entropy)

In [38]:
# 3.6 Compare Human vs. AI
long_df.groupby("label")[[
    "ending_diversity",
    "ending_entropy"
]].describe()

ending_diversity                                         ending_entropy  \
                 count mean       std  min  25%  50%  75%  max          count   
label                                                                           
ai                50.0  0.2  0.404061  0.0  0.0  0.0  0.0  1.0           50.0   
human             50.0  0.2  0.404061  0.0  0.0  0.0  0.0  1.0           50.0   

                                                         
           mean       std  min  25%  50%  75%       max  
label                                                    
ai     0.384935  0.920859 -0.0  0.0  0.0  0.0  3.321928  
human  0.323399  0.792759 -0.0  0.0  0.0  0.0  4.000000

In [39]:
# 3.7 Inspect examples
for i in range(5):
    print("TEXT:", long_df["text"].iloc[i])
    print("ENDINGS:", long_df["normalized_endings"].iloc[i])
    print("DIVERSITY:", long_df["ending_diversity"].iloc[i])
    print("-" * 50)

TEXT: アイドルのライブでしたらそういう人もいるかもしれませんが、バンド系のライブでしょうか？
めちゃくちゃ悪目立ちすると思います。
ENDINGS: ['系のライブでしょうか', '目立ちすると思います']
DIVERSITY: 1.0
--------------------------------------------------
TEXT: 基本的な塗料というのは原液の様に濃く作られているからです。

子供の頃に学校で使った絵の具や、馴染みのあるカルピスをそのまま使ったり、飲む方はいませんね、
必ず水で溶いたり、希釈をして使ったり飲んだりしているわけです。
ENDINGS: ['作られているからです', 'だりしているわけです']
DIVERSITY: 1.0
--------------------------------------------------
TEXT: ＞みなさんは放置子に付き纏われたりした際、付き合ってあげますか？

以前の私なら付き合ってあげたと思います。
でもいろいろあって今ではすっかり放置子とその親が苦手になったので、今の私なら1秒も付き合ってあげません。
すぐに放置子から離れます。
ENDINGS: ['付き合ってあげますか', 'ってあげたと思います', '付き合ってあげません', 'に放置子から離れます']
DIVERSITY: 1.0
--------------------------------------------------
TEXT: EUに加盟することとEURを使うことは同じではありませんが
EU再加盟の機運は高くなっています
ただすぐにどうかは疑問ですが
すくなくても失敗したかも
ヨーロッパもかつてのイギリスの移民問題を理解してくれていそう
というのがあり、若い世代は８割再加盟に賛成のようです

かりに再加盟しEURを使うときのコインはおそらく
表面はEU共通デザイン
裏面は国別なので
国王か王室紋章などでしょうね
ENDINGS: ['室紋章などでしょうね']
DIVERSITY: 1.0
--------------------------------------------------
TEXT: 南アフリカ共和国およびその周辺の国々。

アフリカの大自然の中で人間に保護されながら生息しているビ

# Feature 4: Pronoun + Self-Reference Density (Japanese)
Subjectivity/Reference behavior

This captures how often writers explicitly refer to:

themselves (私 / 僕 / 俺)
the reader (あなた / 君)
general people (人 / 誰か in some contexts)

In Japanese, this is powerful because:
humans often omit subjects
AI often over-explicates them

That mismatch is one of the best signals.


Subject visibility:
Japanese is a pro-drop language, which means that subjects are often implicit.
Thus, when pronouns appear too consistently, it becomes a stylistic signal.

In [40]:
# 4.1 Define pronoun lexicon
pronouns = {
    "私", "わたし", "僕", "ぼく", "俺", "おれ",
    "あなた", "君", "きみ", "お前",
    "人", "誰か", "自分"
}

In [41]:
# 4.2 Sudachi-based pronoun extractor
def extract_pronouns(text):
    if pd.isna(text):
        return []
    
    tokens = tokenizer.tokenize(text)
    
    found = []
    for m in tokens:
        surface = m.surface()
        base = m.dictionary_form()
        
        if surface in pronouns or base in pronouns:
            found.append(surface)
    
    return found

In [42]:
# 4.3 Apply to dataset
long_df["pronouns"] = long_df["text"].apply(extract_pronouns)

In [43]:
# 4.4 Count pronouns
long_df["pronoun_count"] = long_df["pronouns"].apply(len)

In [44]:
# 4.5 Normalize for length
# Because longer answers naturally have more pronouns

long_df["pronoun_density"] = long_df["pronoun_count"] / long_df["token_count"].replace(0, 1)

In [45]:
# 4.6 Compare human vs. AI
long_df.groupby("label")[[
    "pronoun_count",
    "pronoun_density"
]].describe()

pronoun_count                                          pronoun_density  \
              count  mean       std  min  25%  50%  75%  max           count   
label                                                                          
ai             50.0  0.10  0.364216  0.0  0.0  0.0  0.0  2.0            50.0   
human          50.0  0.12  0.435187  0.0  0.0  0.0  0.0  2.0            50.0   

                                                         
           mean       std  min  25%  50%  75%       max  
label                                                    
ai     0.000843  0.003289  0.0  0.0  0.0  0.0  0.019608  
human  0.002049  0.007582  0.0  0.0  0.0  0.0  0.037037

In [46]:
# 4.7 Inspect examples
for i in range(5):
    print("TEXT:", long_df["text"].iloc[i])
    print("PRONOUNS:", long_df["pronouns"].iloc[i])
    print("COUNT:", long_df["pronoun_count"].iloc[i])
    print("-" * 50)

TEXT: アイドルのライブでしたらそういう人もいるかもしれませんが、バンド系のライブでしょうか？
めちゃくちゃ悪目立ちすると思います。
PRONOUNS: ['人']
COUNT: 1
--------------------------------------------------
TEXT: 基本的な塗料というのは原液の様に濃く作られているからです。

子供の頃に学校で使った絵の具や、馴染みのあるカルピスをそのまま使ったり、飲む方はいませんね、
必ず水で溶いたり、希釈をして使ったり飲んだりしているわけです。
PRONOUNS: []
COUNT: 0
--------------------------------------------------
TEXT: ＞みなさんは放置子に付き纏われたりした際、付き合ってあげますか？

以前の私なら付き合ってあげたと思います。
でもいろいろあって今ではすっかり放置子とその親が苦手になったので、今の私なら1秒も付き合ってあげません。
すぐに放置子から離れます。
PRONOUNS: ['私', '私']
COUNT: 2
--------------------------------------------------
TEXT: EUに加盟することとEURを使うことは同じではありませんが
EU再加盟の機運は高くなっています
ただすぐにどうかは疑問ですが
すくなくても失敗したかも
ヨーロッパもかつてのイギリスの移民問題を理解してくれていそう
というのがあり、若い世代は８割再加盟に賛成のようです

かりに再加盟しEURを使うときのコインはおそらく
表面はEU共通デザイン
裏面は国別なので
国王か王室紋章などでしょうね
PRONOUNS: []
COUNT: 0
--------------------------------------------------
TEXT: 南アフリカ共和国およびその周辺の国々。

アフリカの大自然の中で人間に保護されながら生息しているビッグ5(ヒョウ・ライオン・サイ・アフリカゾウ・カバ)を観たい。またビクトリア大瀑布を見学したいから。
PRONOUNS: []
COUNT: 0
--------------------------------------------------


# Feature 5: Kana / Kanji ratio (orthographic signature)
This measures lexical + stylistic encoding preference

Not the strongest feature, as Kana/Kanji ratio is a mostly shallow orthographic choice, which means that the data can be noisy.
It is heavily influenced by:
- topic domain
- platform norms
- user typing habits
- IME suggestions

However, this is subtle but somehwat informative in Japanese writing systems:
- AI often produces more balanced kanji usage
- humans show stronger stylistic drift depending on writing habit

Even though everything is typed (not hand-written), people still differ in:
- kanji preference (教育 vs きょういく)
- simplification habits (難しい vs むずかしい)
- stylistic readability choices
- domain familiarity

In [1]:
# 5.1 Define helper functions
import unicodedata

# Classify Kana/Kanji based on unicode ranges:
def is_kanji(char):
    return '\u4e00' <= char <= '\u9fff'

def is_kana(char):
    return (
        '\u3040' <= char <= '\u309f' or  # hiragana
        '\u30a0' <= char <= '\u30ff'     # katakana
    )

In [2]:
# 5.2 Compute ratios
def kana_kanji_ratio(text):
    if pd.isna(text):
        return pd.Series([0, 0, 0])
    
    kanji = sum(is_kanji(c) for c in text)
    kana = sum(is_kana(c) for c in text)
    total = len(text) if len(text) > 0 else 1
    
    return pd.Series([
        kanji / total,
        kana / total,
        kanji / (kana + 1)  # avoid division by zero
    ])

In [50]:
# 5.3 Apply to dataset
long_df[["kanji_ratio", "kana_ratio", "kanji_to_kana"]] = (
    long_df["text"].apply(kana_kanji_ratio)
)

In [51]:
# 5.4 Compare human vs. AI
long_df.groupby("label")[[
    "kanji_ratio",
    "kana_ratio",
    "kanji_to_kana"
]].describe()

kanji_ratio                                                    \
            count      mean       std  min  25%  50%  75%       max   
label                                                                 
ai           50.0  0.062305  0.128775  0.0  0.0  0.0  0.0  0.426346   
human        50.0  0.046923  0.097746  0.0  0.0  0.0  0.0  0.290698   

      kana_ratio            ...                kanji_to_kana            \
           count      mean  ...  75%       max         count      mean   
label                       ...                                          
ai          50.0  0.118452  ...  0.0  0.652695          50.0  0.107003   
human       50.0  0.130136  ...  0.0  0.825397          50.0  0.073846   

                                               
            std  min  25%  50%  75%       max  
label                                          
ai     0.228264  0.0  0.0  0.0  0.0  0.895833  
human  0.157280  0.0  0.0  0.0  0.0  0.556075  

[2 rows x 24 columns]

In [52]:
# 5.5 Quick inspection
for i in range(5):
    print("TEXT:", long_df["text"].iloc[i])
    print("KANJI RATIO:", long_df["kanji_ratio"].iloc[i])
    print("KANA RATIO:", long_df["kana_ratio"].iloc[i])
    print("-" * 50)

TEXT: アイドルのライブでしたらそういう人もいるかもしれませんが、バンド系のライブでしょうか？
めちゃくちゃ悪目立ちすると思います。
KANJI RATIO: 0.09523809523809523
KANA RATIO: 0.8253968253968254
--------------------------------------------------
TEXT: 基本的な塗料というのは原液の様に濃く作られているからです。

子供の頃に学校で使った絵の具や、馴染みのあるカルピスをそのまま使ったり、飲む方はいませんね、
必ず水で溶いたり、希釈をして使ったり飲んだりしているわけです。
KANJI RATIO: 0.26785714285714285
KANA RATIO: 0.625
--------------------------------------------------
TEXT: ＞みなさんは放置子に付き纏われたりした際、付き合ってあげますか？

以前の私なら付き合ってあげたと思います。
でもいろいろあって今ではすっかり放置子とその親が苦手になったので、今の私なら1秒も付き合ってあげません。
すぐに放置子から離れます。
KANJI RATIO: 0.23809523809523808
KANA RATIO: 0.626984126984127
--------------------------------------------------
TEXT: EUに加盟することとEURを使うことは同じではありませんが
EU再加盟の機運は高くなっています
ただすぐにどうかは疑問ですが
すくなくても失敗したかも
ヨーロッパもかつてのイギリスの移民問題を理解してくれていそう
というのがあり、若い世代は８割再加盟に賛成のようです

かりに再加盟しEURを使うときのコインはおそらく
表面はEU共通デザイン
裏面は国別なので
国王か王室紋章などでしょうね
KANJI RATIO: 0.22815533980582525
KANA RATIO: 0.6019417475728155
--------------------------------------------------
TEXT: 南アフリカ共和国およびその周辺の国々。

アフリカの大自然の中で人